### chromaDb

In [17]:
import os
import dotenv
from chromadb.utils import embedding_functions
import chromadb
dotenv.load_dotenv()

client = chromadb.Client()

ali_ef = embedding_functions.OpenAIEmbeddingFunction(
    model_name="tongyi-embedding-vision-plus-2026-03-06",  # 阿里向量模型
    api_key= os.getenv("DASHSCOPE_API_KEY"),
    api_base="https://dashscope.aliyuncs.com/compatible-mode/v1"
)
collection = client.create_collection(name="demo_collection1")


In [11]:
print(collection.count())
# print(f"Collection名称: {collection.name}")
print(f"Collection名称: {collection}")

0
Collection名称: Collection(name=demo_collection)


In [12]:
documents = [
    "苹果是一种水果，富含维生素",
    "Python是一种编程语言",
    "机器学习算法可以预测未来趋势",
    "香蕉含有丰富的钾元素",
    "深度学习是机器学习的子集"
]

ids = [f"doc_{i}" for i in range(len(documents))]
metadatas = [
    {"category": "食物"},
    {"category": "技术"},
    {"category": "技术"},
    {"category": "食物"},
    {"category": "技术"}
]

# 4. 添加文档
add = collection.add(documents=documents, ids=ids, metadatas=metadatas)
print(add)

NotFoundError: Error code: 404 - {'error': {'message': 'Unsupported model `tongyi-embedding-vision-plus-2026-03-06` for OpenAI compatibility mode.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_supported'}, 'request_id': '58dc7cbb-e681-9695-aed0-8f67b466f9e6'} in add.

In [19]:
class RAGQuestionAnswering:
    def __init__(self,knowledge_base_path="./knowledge_base_db"):
        self.client = chromadb.PersistentClient(knowledge_base_path)
        self.collection = client.get_or_create_collection(
            name="knowledge_base"
        )
    def add_knowledge(self,documents,sources=None):
        """添加知识到知识库"""
        ids = [f"kb_{i}" for i in range(len(documents))]
        metadatas = [{"source": source} for source in (sources or ["unknown"] * len(documents))]
        self.collection.add(documents=documents, ids=ids, metadatas=metadatas)
        
    def query_knowledge(self,question,top_n=3):
        results = self.collection.query(query_texts=[question], n_results=top_n)
        relevant_docs = []
        for doc, distance, metadata in zip(
            results['documents'][0],
            results['distances'][0],
            results['metadatas'][0]
        ):
            relevant_docs.append({
                "content": doc,
                "relevance_score": 1 - distance,
                "source": metadata.get("source", "unknown")
            })
        
        return relevant_docs


rag_system = RAGQuestionAnswering()
rag_system.add_knowledge([
    "Python是一种高级编程语言",
    "机器学习是人工智能的分支",
    "Chroma是向量数据库"
], ["python_doc", "ml_doc", "chroma_doc"])

knowledge = rag_system.query_knowledge("什么是python")
for item in knowledge:
     print(f"- {item['content']} (相关性: {item['relevance_score']:.3f})")


- Python是一种高级编程语言 (相关性: 0.484)
- Chroma是向量数据库 (相关性: -0.018)
- 机器学习是人工智能的分支 (相关性: -0.268)
